# Figure 3: IPC Phase Classification Flowchart

## Purpose
Generate a **Graphviz flowchart** illustrating the IPC (Integrated Food Security Phase Classification) 20% threshold decision rule.

## Methodology
1. Use `graphviz` Python library to create decision tree diagram
2. Show hierarchical classification logic (Phase 5 → 4 → 3 → 2 → 1)
3. Visualize 20% threshold rule at each decision node

## Dependencies
- `graphviz` Python package (Python bindings)
- Graphviz system executables (must be installed separately - see INSTALL.md)

## Expected Outputs
- PNG/PDF flowchart diagram for manuscript Figure 3
- Visual representation of phase classification methodology

## Note
⚠️ **SYSTEM REQUIREMENT**: Ensure Graphviz is installed on your system:
- Windows: Download from graphviz.org
- macOS: `brew install graphviz`
- Linux: `sudo apt-get install graphviz`

Verify with: `dot -V`

In [ ]:
import graphviz
import textwrap

# --- Configuration (Adjusted Spacing & BACKGROUND COLOR) ---
graph_attr = {
    'rankdir': 'TB',
    'splines': 'ortho',
    'nodesep': '0.4',
    'ranksep': '0.5',
    'fontname': 'Arial',
    'fontsize': '10',
    'bgcolor': 'white',  # Set background to white
    'dpi': '300' # Optional: Set DPI if needed for JPG quality
}
# --- Node, Edge attributes, Colors, Annotation remain the same ---
node_attr = {
    'fontname': 'Arial',
    'fontsize': '9',
    'style': 'filled,rounded',
    'color': 'black',
    'fontcolor': 'black',
    'shape': 'box'
}
edge_attr = {
    'fontname': 'Arial',
    'fontsize': '8',
    'color': 'black',
    'arrowsize': '0.7',
}
color_regressor = '#D6EAF8'
color_prob = '#D5F5E3'
color_diamond = '#E0E0E0'
color_phase = '#AED6F1'
color_cluster_box = '#F0F0F0'
color_cluster_border = '#A0A0A0'
annotation_raw = (
    "This flowchart outlines the phase classification process, adapting the top-down hierarchical "
    "approach of the Integrated Food Security Phase Classification (IPC). Predicted probabilities for "
    "each phase (or worse) are evaluated sequentially against a 0.2 threshold (representing the '20% rule'). "
    "Classification stops at the most severe phase meeting the threshold.\n\n"
    "Source: Adapted from IPC Technical Manual Version 3.0 (2019) - https://www.ipcinfo.org/fileadmin/user_upload/ipcinfo/manual/IPC_Technical_Manual_3_Final.pdf"
)
annotation_wrapped = '\n'.join(textwrap.wrap(annotation_raw, width=120))

# --- Create Flowchart ---
dot = graphviz.Digraph(comment='IPC Phase Classification Flowchart - Final JPG')
dot.attr(**graph_attr)
dot.attr(label=annotation_wrapped, labelloc='b', fontsize='8')
dot.attr('node', **node_attr)
dot.attr('edge', **edge_attr)

# --- Define Nodes within Clusters (Unchanged) ---
with dot.subgraph(name='cluster_phase5') as c5:
    c5.attr(label='', color=color_cluster_border, style='filled', fillcolor=color_cluster_box)
    c5.node('r5', 'XGBoost:\nPredict % Pop Phase 5', fillcolor=color_regressor)
    c5.node('pr5', 'Predicted % Pop\nPhase 5', shape='ellipse', style='filled', fillcolor=color_prob)
with dot.subgraph(name='cluster_phase4') as c4:
    c4.attr(label='', color=color_cluster_border, style='filled', fillcolor=color_cluster_box)
    c4.node('r4', 'XGBoost:\nPredict % Pop Phase 4+', fillcolor=color_regressor)
    c4.node('pr4', 'Predicted % Pop\nPhase 4+', shape='ellipse', style='filled', fillcolor=color_prob)
with dot.subgraph(name='cluster_phase3') as c3:
    c3.attr(label='', color=color_cluster_border, style='filled', fillcolor=color_cluster_box)
    c3.node('r3', 'XGBoost:\nPredict % Pop Phase 3+', fillcolor=color_regressor)
    c3.node('pr3', 'Predicted % Pop\nPhase 3+', shape='ellipse', style='filled', fillcolor=color_prob)
with dot.subgraph(name='cluster_phase2') as c2:
    c2.attr(label='', color=color_cluster_border, style='filled', fillcolor=color_cluster_box)
    c2.node('r2', 'XGBoost:\nPredict % Pop Phase 2+', fillcolor=color_regressor)
    c2.node('pr2', 'Predicted % Pop\nPhase 2+', shape='ellipse', style='filled', fillcolor=color_prob)

# --- Define Decision and Phase Nodes (Unchanged) ---
dot.node('d5', '>=20%?', shape='diamond', style='filled', fillcolor=color_diamond, rounded='false')
dot.node('d4', '>=20%?', shape='diamond', style='filled', fillcolor=color_diamond, rounded='false')
dot.node('d3', '>=20%?', shape='diamond', style='filled', fillcolor=color_diamond, rounded='false')
dot.node('d2', '>=20%?', shape='diamond', style='filled', fillcolor=color_diamond, rounded='false')
dot.node('p5', 'Phase 5', fillcolor=color_phase)
dot.node('p4', 'Phase 4', fillcolor=color_phase)
dot.node('p3', 'Phase 3', fillcolor=color_phase)
dot.node('p2', 'Phase 2', fillcolor=color_phase)
dot.node('p1', 'Phase 1', fillcolor=color_phase)

# --- Add Rank Constraints (Unchanged) ---
with dot.subgraph() as s:
    s.attr(rank='same')
    s.node('d5'); s.node('d4'); s.node('d3'); s.node('d2')
with dot.subgraph() as s:
    s.attr(rank='same')
    s.node('p5'); s.node('p4'); s.node('p3'); s.node('p2')

# --- Define Edges (Unchanged) ---
dot.edge('r5', 'pr5')
dot.edge('r4', 'pr4')
dot.edge('r3', 'pr3')
dot.edge('r2', 'pr2')
dot.edge('pr5', 'd5')
dot.edge('pr4', 'd4')
dot.edge('pr3', 'd3')
dot.edge('pr2', 'd2')
dot.edge('d5', 'p5', label=' Yes')
dot.edge('d4', 'p4', label=' Yes')
dot.edge('d3', 'p3', label=' Yes')
dot.edge('d2', 'p2', label='    Yes')
dot.edge('d2', 'p1', label='No')
dot.edge('d5', 'd4', taillabel='No')
dot.edge('d4', 'd3', taillabel='No')
dot.edge('d3', 'd2', taillabel='No')

# --- Render and Save ---
output_filename = 'ipc_flowchart_final_nature'
output_format = 'jpg' # Set output format to JPG

try:
    # You can also set DPI directly in the render call if needed
    dot.graph_attr.update(dpi='300')
    dot.render(output_filename, view=False, format=output_format, cleanup=True)
    print(f"Flowchart saved as {output_filename}.{output_format}")
except graphviz.exceptions.ExecutableNotFound:
    print("ERROR: Graphviz executable not found.")
    print("Please install Graphviz from https://graphviz.org/download/ and ensure it's in your system's PATH.")
except Exception as e:
    print(f"An error occurred during rendering: {e}")

In [ ]:
import graphviz
import textwrap

# --- Configuration ---
graph_attr = {
    'rankdir': 'TB',
    'splines': 'ortho',
    'nodesep': '0.4',
    'ranksep': '0.5',
    'fontname': 'Arial',
    'fontsize': '10', # Ensure fontsize is a string if required by your Graphviz version
    'bgcolor': 'white',
    'dpi': '300' # Ensure DPI is a string
}
node_attr = {
    'fontname': 'Arial',
    'fontsize': '9', # Ensure fontsize is a string
    'style': 'filled,rounded',
    'color': 'black', # Border color for nodes
    'fontcolor': 'black',
    'shape': 'box'
}
edge_attr = {
    'fontname': 'Arial',
    'fontsize': '8', # Ensure fontsize is a string
    'color': 'black',
    'arrowsize': '0.7',
}
color_regressor = '#D6EAF8'
color_prob = '#D5F5E3'
color_diamond = '#E0E0E0'
color_phase = '#AED6F1'
color_cluster_box = '#F0F0F0'
color_cluster_border = '#A0A0A0'

# For the new rules cluster box
color_rules_cluster_border = 'black' # Border for the transparent rectangle


# --- Create Flowchart ---
dot = graphviz.Digraph(comment='IPC Phase Classification Flowchart - Rules Cluster')
dot.attr(**graph_attr)
dot.attr('node', **node_attr)
dot.attr('edge', **edge_attr)

# --- Define Nodes within top-level Clusters (XGBoost & Predicted Pop) ---
with dot.subgraph(name='cluster_phase5') as c5:
    c5.attr(label='', color=color_cluster_border, style='filled', fillcolor=color_cluster_box)
    c5.node('r5', 'XGBoost:\nPredict % Pop Phase 5', fillcolor=color_regressor)
    c5.node('pr5', 'Predicted % Pop\nPhase 5', shape='ellipse', style='filled', fillcolor=color_prob)
with dot.subgraph(name='cluster_phase4') as c4:
    c4.attr(label='', color=color_cluster_border, style='filled', fillcolor=color_cluster_box)
    c4.node('r4', 'XGBoost:\nPredict % Pop Phase 4+', fillcolor=color_regressor)
    c4.node('pr4', 'Predicted % Pop\nPhase 4+', shape='ellipse', style='filled', fillcolor=color_prob)
with dot.subgraph(name='cluster_phase3') as c3:
    c3.attr(label='', color=color_cluster_border, style='filled', fillcolor=color_cluster_box)
    c3.node('r3', 'XGBoost:\nPredict % Pop Phase 3+', fillcolor=color_regressor)
    c3.node('pr3', 'Predicted % Pop\nPhase 3+', shape='ellipse', style='filled', fillcolor=color_prob)
with dot.subgraph(name='cluster_phase2') as c2:
    c2.attr(label='', color=color_cluster_border, style='filled', fillcolor=color_cluster_box)
    c2.node('r2', 'XGBoost:\nPredict % Pop Phase 2+', fillcolor=color_regressor)
    c2.node('pr2', 'Predicted % Pop\nPhase 2+', shape='ellipse', style='filled', fillcolor=color_prob)

# --- Define Decision and Phase Nodes (Globally, with their specific styles) ---
dot.node('d5', '>=20%?', shape='diamond', style='filled', fillcolor=color_diamond, rounded='false')
dot.node('d4', '>=20%?', shape='diamond', style='filled', fillcolor=color_diamond, rounded='false')
dot.node('d3', '>=20%?', shape='diamond', style='filled', fillcolor=color_diamond, rounded='false')
dot.node('d2', '>=20%?', shape='diamond', style='filled', fillcolor=color_diamond, rounded='false')

dot.node('p5', 'Phase 5', fillcolor=color_phase)
dot.node('p4', 'Phase 4', fillcolor=color_phase)
dot.node('p3', 'Phase 3', fillcolor=color_phase)
dot.node('p2', 'Phase 2', fillcolor=color_phase)
dot.node('p1', 'Phase 1', fillcolor=color_phase)


# --- MODIFICATION: Cluster for IPC classification rules (diamond boxes) ---
with dot.subgraph(name='cluster_rules') as cr:
    cr.attr(label='IPC classification rules',
            fontname='Arial',
            fontsize='10', # Ensure string
            style='filled,rounded',    # Apply style for fillcolor and rounded corners
            fillcolor='transparent', # Make the background transparent
            color=color_rules_cluster_border, # Border color of the rectangle
            margin='20',             # Padding around the nodes within this cluster (string)
            labelloc='t',            # Position label at the top of the cluster
            labeljust='c'            # Center the label
            )
    # This internal subgraph ensures the diamond nodes are ranked the same,
    # and associates them with 'cluster_rules'.
    # It refers to the d5, d4, d3, d2 nodes already defined globally.
    with cr.subgraph() as s_rank_diamonds:
        s_rank_diamonds.attr(rank='same')
        s_rank_diamonds.node('d5')
        s_rank_diamonds.node('d4')
        s_rank_diamonds.node('d3')
        s_rank_diamonds.node('d2')
# --- End of MODIFICATION ---

# --- Add Rank Constraints for the lower Phase Nodes (p2-p5) ---
# This was the original setup for these phase nodes.
with dot.subgraph() as s_phases:
    s_phases.attr(rank='same')
    s_phases.node('p5') # Refers to already defined p5
    s_phases.node('p4') # Refers to already defined p4
    s_phases.node('p3') # Refers to already defined p3
    s_phases.node('p2') # Refers to already defined p2

# --- Define Edges (Should remain unchanged in behavior) ---
dot.edge('r5', 'pr5')
dot.edge('r4', 'pr4')
dot.edge('r3', 'pr3')
dot.edge('r2', 'pr2')

dot.edge('pr5', 'd5')
dot.edge('pr4', 'd4')
dot.edge('pr3', 'd3')
dot.edge('pr2', 'd2')

dot.edge('d5', 'p5', label=' Yes')
dot.edge('d4', 'p4', label=' Yes')
dot.edge('d3', 'p3', label=' Yes')
dot.edge('d2', 'p2', label='     Yes') # Original spacing for alignment

dot.edge('d5', 'd4', taillabel='No')
dot.edge('d4', 'd3', taillabel='No')
dot.edge('d3', 'd2', taillabel='No')
dot.edge('d2', 'p1', label='No')

# --- Render and Save ---
output_filename = 'ipc_flowchart_rules_transparent_cluster' # Changed filename for the new output
output_format = 'jpg'

try:
    dot.graph_attr.update(dpi=str(graph_attr['dpi'])) # Ensure DPI is passed as string
    dot.render(output_filename, view=False, format=output_format, cleanup=True)
    print(f"Flowchart saved as {output_filename}.{output_format}")
except graphviz.exceptions.ExecutableNotFound:
    print("ERROR: Graphviz executable not found.")
    print("Please install Graphviz from https://graphviz.org/download/ and ensure it's in your system's PATH.")
except Exception as e:
    print(f"An error occurred during rendering: {e}")